<a href="https://colab.research.google.com/github/gabiuxo/Algoritmos-de-Aprendizaje-Automatico/blob/main/Practica_Tema_6_Ensambles_de_Modelos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Reto 01 — El comité cambia de opinión (Exploración)

This challenge focuses on understanding how altering hyperparameters of base models in an ensemble affects the final aggregation and diversity of predictions, specifically using the `breast_cancer` dataset.

In [3]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import roc_auc_score

# Load the dataset
data = load_breast_cancer()
X, y = data.data, data.target

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")

Shape of X_train: (398, 30)
Shape of X_test: (171, 30)


### Initial Ensemble Setup

As per the challenge, we will set up an initial `VotingClassifier` with `LogisticRegression`, `KNeighborsClassifier`, and `DecisionTreeClassifier` using soft voting.

In [4]:
# Initialize base models with fixed random_state for reproducibility
clf1 = LogisticRegression(random_state=42, solver='liblinear') # solver='liblinear' for small datasets
clf2 = KNeighborsClassifier(n_neighbors=5) # Default n_neighbors is 5
clf3 = DecisionTreeClassifier(random_state=42, max_depth=None) # Default max_depth is None

# Create the VotingClassifier (soft voting as specified in the context of the PDF)
eclf = VotingClassifier(estimators=[('lr', clf1), ('knn', clf2), ('dt', clf3)], voting='soft', n_jobs=-1)
eclf = eclf.fit(X_train, y_train)
y_pred_proba = eclf.predict_proba(X_test)[:, 1]
roc_auc_initial = roc_auc_score(y_test, y_pred_proba)

print(f"Initial Ensemble ROC-AUC: {roc_auc_initial:.4f}")

Initial Ensemble ROC-AUC: 0.9971


### Reto 01 - Parte 1: Cambiar `n_neighbors` en `KNeighborsClassifier`

Now, let's change the `n_neighbors` parameter in `KNeighborsClassifier` to 3 and then to 15, retraining the ensemble and calculating the ROC-AUC for each case.

In [5]:
# n_neighbors = 3
clf2_n3 = KNeighborsClassifier(n_neighbors=3)
eclf_n3 = VotingClassifier(estimators=[('lr', clf1), ('knn', clf2_n3), ('dt', clf3)], voting='soft', n_jobs=-1)
eclf_n3 = eclf_n3.fit(X_train, y_train)
y_pred_proba_n3 = eclf_n3.predict_proba(X_test)[:, 1]
roc_auc_n3 = roc_auc_score(y_test, y_pred_proba_n3)
print(f"Ensemble ROC-AUC with n_neighbors=3: {roc_auc_n3:.4f}")

# n_neighbors = 15
clf2_n15 = KNeighborsClassifier(n_neighbors=15)
eclf_n15 = VotingClassifier(estimators=[('lr', clf1), ('knn', clf2_n15), ('dt', clf3)], voting='soft', n_jobs=-1)
eclf_n15 = eclf_n15.fit(X_train, y_train)
y_pred_proba_n15 = eclf_n15.predict_proba(X_test)[:, 1]
roc_auc_n15 = roc_auc_score(y_test, y_pred_proba_n15)
print(f"Ensemble ROC-AUC with n_neighbors=15: {roc_auc_n15:.4f}")

Ensemble ROC-AUC with n_neighbors=3: 0.9965
Ensemble ROC-AUC with n_neighbors=15: 0.9971


### Reto 01 - Parte 2: Cambiar `max_depth` del `DecisionTreeClassifier`

Next, we will change the `max_depth` parameter of the `DecisionTreeClassifier` to 2 and then to 10, retraining the ensemble and measuring the ROC-AUC for each.

In [6]:
# max_depth = 2
clf3_d2 = DecisionTreeClassifier(random_state=42, max_depth=2)
eclf_d2 = VotingClassifier(estimators=[('lr', clf1), ('knn', clf2), ('dt', clf3_d2)], voting='soft', n_jobs=-1)
eclf_d2 = eclf_d2.fit(X_train, y_train)
y_pred_proba_d2 = eclf_d2.predict_proba(X_test)[:, 1]
roc_auc_d2 = roc_auc_score(y_test, y_pred_proba_d2)
print(f"Ensemble ROC-AUC with max_depth=2: {roc_auc_d2:.4f}")

# max_depth = 10
clf3_d10 = DecisionTreeClassifier(random_state=42, max_depth=10)
eclf_d10 = VotingClassifier(estimators=[('lr', clf1), ('knn', clf2), ('dt', clf3_d10)], voting='soft', n_jobs=-1)
eclf_d10 = eclf_d10.fit(X_train, y_train)
y_pred_proba_d10 = eclf_d10.predict_proba(X_test)[:, 1]
roc_auc_d10 = roc_auc_score(y_test, y_pred_proba_d10)
print(f"Ensemble ROC-AUC with max_depth=10: {roc_auc_d10:.4f}")

Ensemble ROC-AUC with max_depth=2: 0.9972
Ensemble ROC-AUC with max_depth=10: 0.9971


## Reto 02 — Voto duro vs. voto suave, en la práctica (Aplicación)

This challenge aims to differentiate operationally and conceptually between hard voting (aggregates discrete labels) and soft voting (aggregates probabilities), recognizing which evaluation metrics are compatible with each strategy.

We will use the same base models (`clf1`, `clf2`, `clf3`) from Reto 01.

In [7]:
from sklearn.metrics import accuracy_score

# Create a copy of the base ensemble configured with voting='hard'
eclf_hard = VotingClassifier(estimators=[('lr', clf1), ('knn', clf2), ('dt', clf3)], voting='hard', n_jobs=-1)

# Train both versions (soft and hard) on the same X_train/y_train
# The soft voting ensemble (eclf) is already trained from Reto 01, but we'll retrain for clarity.
eclf_soft = VotingClassifier(estimators=[('lr', clf1), ('knn', clf2), ('dt', clf3)], voting='soft', n_jobs=-1)
eclf_soft.fit(X_train, y_train)
eclf_hard.fit(X_train, y_train)

# Compare the accuracy_score of both versions on X_test
y_pred_soft = eclf_soft.predict(X_test)
y_pred_hard = eclf_hard.predict(X_test)

accuracy_soft = accuracy_score(y_test, y_pred_soft)
accuracy_hard = accuracy_score(y_test, y_pred_hard)

print(f"Accuracy Score (Soft Voting): {accuracy_soft:.4f}")
print(f"Accuracy Score (Hard Voting): {accuracy_hard:.4f}")

Accuracy Score (Soft Voting): 0.9825
Accuracy Score (Hard Voting): 0.9766


### Reto 02 - Parte 2: Intentar calcular ROC-AUC de la versión 'hard'

Now, let's try to calculate the ROC-AUC for the hard voting ensemble. The challenge asks us to observe what happens and explain it conceptually.

In [8]:
try:
    y_pred_proba_hard = eclf_hard.predict_proba(X_test)[:, 1]
    roc_auc_hard = roc_auc_score(y_test, y_pred_proba_hard)
    print(f"ROC-AUC Score (Hard Voting): {roc_auc_hard:.4f}")
except AttributeError as e:
    print(f"Error calculating ROC-AUC for Hard Voting: {e}")
    print("This error occurs because 'VotingClassifier' with 'hard' voting does not expose 'predict_proba'.")

Error calculating ROC-AUC for Hard Voting: This 'VotingClassifier' has no attribute 'predict_proba'
This error occurs because 'VotingClassifier' with 'hard' voting does not expose 'predict_proba'.


## Reto 03 — Un cuarto integrante para el comité (Modificación)

This challenge explores whether adding a complex model like `RandomForestClassifier` to an existing ensemble improves performance or merely increases complexity without providing useful diversity to the ensemble's errors.

In [9]:
from sklearn.ensemble import RandomForestClassifier

# Define the new base model
clf4 = RandomForestClassifier(n_estimators=100, random_state=42)

# Agrega un cuarto modelo base al VotingClassifier: un RandomForestClassifier
eclf_4_models = VotingClassifier(estimators=[('lr', clf1), ('knn', clf2), ('dt', clf3), ('rf', clf4)], voting='soft', n_jobs=-1)

# Entrena el ensamble de 4 modelos y calcula su ROC-AUC
eclf_4_models = eclf_4_models.fit(X_train, y_train)
y_pred_proba_4_models = eclf_4_models.predict_proba(X_test)[:, 1]
roc_auc_4_models = roc_auc_score(y_test, y_pred_proba_4_models)

print(f"Initial Ensemble (3 models) ROC-AUC: {roc_auc_initial:.4f}")
print(f"Ensemble with 4 models (LR, KNN, DT, RF) ROC-AUC: {roc_auc_4_models:.4f}")

Initial Ensemble (3 models) ROC-AUC: 0.9971
Ensemble with 4 models (LR, KNN, DT, RF) ROC-AUC: 0.9978


### Reto 03 - Parte 2: Repetir el experimento quitando `KNeighborsClassifier`

Now, we will repeat the experiment but remove the `KNeighborsClassifier`, keeping `LogisticRegression`, `DecisionTreeClassifier`, and `RandomForestClassifier`.

In [10]:
# Quitar el KNeighborsClassifier (deja regresión logística, árbol y random forest)
eclf_3_models_no_knn = VotingClassifier(estimators=[('lr', clf1), ('dt', clf3), ('rf', clf4)], voting='soft', n_jobs=-1)

# Entrena este nuevo ensamble y calcula su ROC-AUC
eclf_3_models_no_knn = eclf_3_models_no_knn.fit(X_train, y_train)
y_pred_proba_3_models_no_knn = eclf_3_models_no_knn.predict_proba(X_test)[:, 1]
roc_auc_3_models_no_knn = roc_auc_score(y_test, y_pred_proba_3_models_no_knn)

print(f"Initial Ensemble (3 models - LR, KNN, DT) ROC-AUC: {roc_auc_initial:.4f}")
print(f"Ensemble with 3 models (LR, DT, RF) ROC-AUC: {roc_auc_3_models_no_knn:.4f}")

Initial Ensemble (3 models - LR, KNN, DT) ROC-AUC: 0.9971
Ensemble with 3 models (LR, DT, RF) ROC-AUC: 0.9966


## Reto 04 — Random Forest contra Gradient Boosting (Integración)

This challenge contrasts bagging (`RandomForestClassifier`) and boosting (`GradientBoostingClassifier`), analyzing how they interact with bias-variance trade-off, their stabilization speed, and the use of the out-of-bag error (`oob_score`).

In [11]:
from sklearn.ensemble import GradientBoostingClassifier

# Entrena un RandomForestClassifier activando el parámetro oob_score=True
rf_oob = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42, n_jobs=-1)
rf_oob.fit(X_train, y_train)

# Imprime el oob_score_ y compáralo con el ROC-AUC sobre X_test
oob_score = rf_oob.oob_score_
y_pred_proba_rf = rf_oob.predict_proba(X_test)[:, 1]
roc_auc_rf = roc_auc_score(y_test, y_pred_proba_rf)

print(f"RandomForestClassifier OOB Score: {oob_score:.4f}")
print(f"RandomForestClassifier ROC-AUC on X_test: {roc_auc_rf:.4f}")

RandomForestClassifier OOB Score: 0.9548
RandomForestClassifier ROC-AUC on X_test: 0.9968


### Reto 04 - Parte 2: Variar el número de árboles (`n_estimators`)

Now, we will vary the number of trees (`n_estimators` to 10, 50, and 300) for both `RandomForestClassifier` and `GradientBoostingClassifier` (keeping `learning_rate=0.05` for GB) and observe their ROC-AUC scores.

In [12]:
n_estimators_values = [10, 50, 300]

print("\n--- RandomForestClassifier ---")
for n in n_estimators_values:
    rf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred_proba = rf.predict_proba(X_test)[:, 1]
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    print(f"n_estimators={n}: ROC-AUC = {roc_auc:.4f}")

print("\n--- GradientBoostingClassifier ---")
for n in n_estimators_values:
    gb = GradientBoostingClassifier(n_estimators=n, learning_rate=0.05, random_state=42)
    gb.fit(X_train, y_train)
    y_pred_proba = gb.predict_proba(X_test)[:, 1]
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    print(f"n_estimators={n}: ROC-AUC = {roc_auc:.4f}")


--- RandomForestClassifier ---
n_estimators=10: ROC-AUC = 0.9934
n_estimators=50: ROC-AUC = 0.9961
n_estimators=300: ROC-AUC = 0.9971

--- GradientBoostingClassifier ---
n_estimators=10: ROC-AUC = 0.9779
n_estimators=50: ROC-AUC = 0.9938
n_estimators=300: ROC-AUC = 0.9947
